## Requirements

- **Python ≥ 3.11** — rasterio 1.4+ and numpy 2.x require Python 3.11+
- **Packages:** `pip install requests rasterio numpy matplotlib pystac-client pyproj`
- **Network:** HTTP access to `kanopia.org` (STAC API) and `lab.kanopia.org` (COG files)
- **Credentials:** Basic Auth required for COG files — set `COG_USER` / `COG_PASS` in the config cell

# BCI COG Workshop

Work through the 5 exercises below. Each builds on the previous.
Run cells top-to-bottom. Cells marked `# TODO` are for you to complete.

---
**Ask for help anytime!**

In [ ]:
# ── Configuration — do not modify ────────────────────────────────────────────
STAC_API_URL  = "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/"
COLLECTION_ID = "2024_bci"

# Workshop credentials
COG_USER = "panama"
COG_PASS = "panama123"

# Default point of interest: approximate centre of BCI island
# Change these to any lon/lat inside BCI if you want to explore a different spot
DEFAULT_LON = -79.8450
DEFAULT_LAT =   9.1540

print("Config ready.")

In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
import sys, subprocess
pkgs = ["requests", "rasterio", "numpy", "matplotlib", "pystac-client", "pyproj"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
print("Done.")

In [ ]:
# ── Imports + GDAL settings ───────────────────────────────────────────────────
import os, re
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from pyproj import Transformer
from pystac_client import Client
from urllib.parse import urlparse, urlunparse

os.environ.setdefault("GDAL_HTTP_TIMEOUT",            "120")
os.environ.setdefault("GDAL_HTTP_MAX_RETRY",          "3")
os.environ.setdefault("GDAL_HTTP_RETRY_DELAY",        "5")
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")

# ── Shared helpers (provided — no need to modify) ─────────────────────────────
def add_auth(url, user, pwd):
    p = urlparse(url)
    if p.username:
        return url
    netloc = f"{user}:{pwd}@{p.netloc}"
    return urlunparse((p.scheme, netloc, p.path, p.params, p.query, p.fragment))

def read_rgb_chip(href, lon, lat, user, pwd, chip_px=512):
    url = add_auth(href, user, pwd)
    with rasterio.open(f"/vsicurl/{url}") as src:
        if src.crs.to_epsg() != 4326:
            t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            xp, yp = t.transform(lon, lat)
        else:
            xp, yp = lon, lat
        row, col = src.index(xp, yp)
        half = chip_px // 2
        win  = Window(col - half, row - half, chip_px, chip_px)
        win  = win.intersection(Window(0, 0, src.width, src.height))
        bands = min(src.count, 3)
        return src.read(list(range(1, bands + 1)), window=win), src.crs

def norm_display(band):
    b = band.astype(float)
    lo, hi = np.nanpercentile(b, 2), np.nanpercentile(b, 98)
    return np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)

def to_rgb(arr):
    return np.stack([norm_display(arr[0]), norm_display(arr[1]), norm_display(arr[2])], axis=-1)

print("Imports and helpers ready.")

---
## Exercise 1 — Query the STAC API

**Goal:** connect to the Kanopia STAC API and list all BCI RGB COG assets with their dates.

**Hint:** use `Client.open(STAC_API_URL)` then `.search(collections=[COLLECTION_ID], max_items=500)`.

In [ ]:
# Exercise 1 ──────────────────────────────────────────────────────────────────
# The pattern below matches only the whole-island RGB COG filenames
pattern = re.compile(r"(\d{8})_bciwhole_.*rgb\.cog\.tif")

# TODO: open the STAC client and search the 2024_bci collection
client = ...            # Client.open(STAC_API_URL)
search = ...            # client.search(...)

# item_collection() replaces deprecated get_all_items()
try:
    items = search.item_collection()
except AttributeError:
    items = search.get_all_items()

# TODO: loop over items and their assets; collect matching COG hrefs into cog_list
# Each entry should be a dict: {'date': int(YYYYMMDD), 'href': str, 'item_id': str}
# Only keep assets whose URI contains "/share/1"
cog_list = []
for item in items:
    assets  = item.assets if hasattr(item, 'assets') else item.get('assets', {})
    item_id = getattr(item, 'id', None) or item.get('id')
    for key, asset in (assets.items() if isinstance(assets, dict) else []):
        href = asset.href if hasattr(asset, 'href') else asset.get('href', '')
        m = pattern.search(href)
        if m and "/share/1" in href:   # keep only publicly accessible assets
            # TODO: append a dict to cog_list
            pass

cog_list.sort(key=lambda x: x['date'])

# Expected: a list of dicts, one per flight date, sorted oldest → newest
print(f"Found {len(cog_list)} BCI whole-island RGB COGs:")
for c in cog_list:
    d = str(c['date'])
    print(f"  {d[:4]}-{d[4:6]}-{d[6:]}  {c['item_id']}")

---
## Exercise 2 — Open a COG and display an RGB chip

**Goal:** read a 512×512 px window from one COG and show it as an RGB image.

**Hints:**
- Use `read_rgb_chip(href, lon, lat, COG_USER, COG_PASS)` (provided above).
- Use `to_rgb(arr)` to convert the `(3, H, W)` array to a display-ready `(H, W, 3)` array.
- COG credentials must be embedded in the URL — `add_auth(href, COG_USER, COG_PASS)` does this.

In [ ]:
# Exercise 2 ──────────────────────────────────────────────────────────────────
# Pick the first date
target = cog_list[0]
d      = str(target['date'])
label  = f"{d[:4]}-{d[4:6]}-{d[6:]}"

print(f"Reading chip from {label} ...")
print(f"File: {target['href'].split('/')[-1]}")

# TODO: call read_rgb_chip to get the (3, H, W) array
arr, cog_crs = ...     # read_rgb_chip(target['href'], DEFAULT_LON, DEFAULT_LAT, COG_USER, COG_PASS)

# TODO: display as RGB image
plt.figure(figsize=(6, 6))
plt.imshow(...)        # to_rgb(arr)
plt.title(f"RGB chip — {label}")
plt.axis('off')
plt.show()

print(f"Chip shape: {arr.shape}  (bands, rows, cols)")
print(f"COG CRS: {cog_crs}")

---
## Exercise 3 — Compute and plot the Green Leaf Index (GLI)

**Formula:** `GLI = (2G − R − B) / (2G + R + B)`

- High GLI (green in RdYlGn) → healthy vegetation
- Low GLI (red in RdYlGn) → dead/bare/stressed

**Hint:** extract `R = arr[0]`, `G = arr[1]`, `B = arr[2]` and work with `float` arrays.

In [ ]:
# Exercise 3 ──────────────────────────────────────────────────────────────────
eps = 1e-6
R, G, B = arr[0].astype(float), arr[1].astype(float), arr[2].astype(float)

# TODO: compute GLI
GLI = ...              # (2*G - R - B) / (2*G + R + B + eps)

# Plot side by side: RGB | GLI
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(to_rgb(arr))
ax1.set_title(f"RGB — {label}")
ax1.axis('off')

# TODO: plot GLI with cmap='RdYlGn', vmin=-0.5, vmax=0.5
im = ax2.imshow(...)   # GLI, cmap=..., vmin=..., vmax=...
ax2.set_title("GLI — Green Leaf Index")
ax2.axis('off')
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"GLI range: [{GLI.min():.3f}, {GLI.max():.3f}]")

---
## Exercise 4 — Compute VARI and GCC; compare all three indices

| Index | Formula |
|-------|---------|
| **VARI** | `(G − R) / (G + R − B)` — clip to [-1.5, 1.5] |
| **GCC**  | `G / (R + G + B)` |

**Goal:** make a 1×4 subplot: RGB, GLI, VARI, GCC.

In [ ]:
# Exercise 4 ──────────────────────────────────────────────────────────────────
# TODO: compute VARI and GCC
VARI = ...             # (G - R) / (G + R - B + eps) , then np.clip(..., -1.5, 1.5)
GCC  = ...             # G / (R + G + B + eps)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(to_rgb(arr))
axes[0].set_title(f"RGB\n{label}")
axes[0].axis('off')

specs = [
    (axes[1], GLI,  "GLI",  "RdYlGn", -0.5,  0.5),
    (axes[2], VARI, "VARI", "RdYlGn", -0.5,  0.8),
    (axes[3], GCC,  "GCC",  "RdYlGn",  0.28, 0.45),
]

for ax, index, name, cmap, vmin, vmax in specs:
    # TODO: plot each index
    im = ax.imshow(...)    # index, cmap=cmap, vmin=vmin, vmax=vmax
    ax.set_title(name)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"BCI vegetation indices — {label}  |  Red=stressed  Green=healthy",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercise 5 (stretch) — Compare GLI across two dates

**Goal:** read the same chip from the **first** and **last** available dates.
Plot RGB and GLI for each date in a 2×2 grid.

What changed between the two dates?  
Can you spot trees that got greener or redder?

In [ ]:
# Exercise 5 ──────────────────────────────────────────────────────────────────
date_early = cog_list[0]
date_late  = cog_list[-1]

def fmt_date(c):
    d = str(c['date'])
    return f"{d[:4]}-{d[4:6]}-{d[6:]}"

print(f"Reading early chip: {fmt_date(date_early)} ...")
# TODO: read chip for early date
arr_e, _ = ...     # read_rgb_chip(date_early['href'], DEFAULT_LON, DEFAULT_LAT, COG_USER, COG_PASS)

print(f"Reading late  chip: {fmt_date(date_late)} ...")
# TODO: read chip for late date
arr_l, _ = ...     # read_rgb_chip(date_late['href'],  DEFAULT_LON, DEFAULT_LAT, COG_USER, COG_PASS)

# TODO: compute GLI for each date
def compute_gli(a):
    R, G, B = a[0].astype(float), a[1].astype(float), a[2].astype(float)
    return (2*G - R - B) / (2*G + R + B + 1e-6)

gli_e = compute_gli(arr_e)
gli_l = compute_gli(arr_l)

# TODO: make a 2x2 figure:
#   [0,0] RGB early   [0,1] RGB late
#   [1,0] GLI early   [1,1] GLI late
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(to_rgb(arr_e))
axes[0, 0].set_title(f"RGB — {fmt_date(date_early)}")
axes[0, 0].axis('off')

axes[0, 1].imshow(to_rgb(arr_l))
axes[0, 1].set_title(f"RGB — {fmt_date(date_late)}")
axes[0, 1].axis('off')

# TODO: plot gli_e and gli_l with cmap='RdYlGn', vmin=-0.5, vmax=0.5
im1 = axes[1, 0].imshow(...)   # gli_e ...
axes[1, 0].set_title(f"GLI — {fmt_date(date_early)}")
axes[1, 0].axis('off')
plt.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04)

im2 = axes[1, 1].imshow(...)   # gli_l ...
axes[1, 1].set_title(f"GLI — {fmt_date(date_late)}")
axes[1, 1].axis('off')
plt.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

fig.suptitle("BCI — Temporal change in GLI  |  Red=stressed  Green=healthy",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Well done!

You've just:
- Queried a STAC API for drone imagery
- Read sub-second chips from multi-GB COG files over the network
- Computed three RGB vegetation indices
- Detected temporal change in canopy health

**Next steps to explore:**
- Change `DEFAULT_LON` / `DEFAULT_LAT` to explore different parts of BCI
- Try middle dates from `cog_list` to build a time series
- Compute a mean GLI for the whole chip on each date and plot it as a curve